# Unix Shell Tutorial - Protein Publications and Disease Recognition

This is the **sixth tutorial** in a series that demonstrates how shell scripting can be used to perform the tasks that health and life science specialists may need to undertake to find and retrieve biomedical data and text. We will use the compound caffeine as an example and explore different public repositories to identify diseases related to it. The focus is not on the specific relationships we may discover, but on the process of obtaining them.

The objective of this tutorial is to learn how to automatically retrieve scientific publications associated with proteins and extract text from their titles and abstracts. Additionally, you will learn how to identify and recognize diseases mentioned within the text.

> This tutorial is part of a series of tutorials adapted as interactive versions of the hands-on steps described in the [Data and Text Processing for Health and Life Sciences](https://labs.rd.ciencias.ulisboa.pt/book/) book, which is licensed under the [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/).

## Step 01 - Publication URL

Now that we have all the PubMed identifiers, we need to download the text included in the titles and abstracts of each publication.

To retrieve from the UniProt citations service the publication entry of a given identifier, we can again use the `curl` command and a link to the publication entry. For example, if we click on the Format button of the UniProt citations service entry, we can get the link to the RDF/XML version. RDF is a standard data model that can be serialized in a XML format. Thus, in our case, we can deal with this format like we did with XML.

To get started, we first need to retrieve the data file generated in the previous tutorial. The following command downloads the chebi_27732_xrefs_UniProt.csv file directly from the GitHub repository:


In [ ]:
%%bash
curl -s -O 'https://raw.githubusercontent.com/lasigeBioTM/data-text-processing-notebooks/refs/heads/main/data/chebi_27732_proteins_xml.zip'
unzip -o chebi_27732_proteins_xml.zip

We can retrieve the publication entry by executing the following command (using a sample PubMed ID):

In [ ]:
%%bash
curl https://rest.uniprot.org/citations/1354642.rdf

**Expected Output:** RDF/XML data for the publication with ID 1354642

> The `curl` command on this platform has access restrictions but works with `rest.uniprot.org` and `eutils.ncbi.nlm.nih.gov` links.

Alternatively, we can use the web service provided by PubMed at NCBI, by still using curl but with another link:

In [ ]:
%%bash
curl 'https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi?db=pubmed&id=1354642&retmode=text&rettype=xml'

**Expected Output:** XML data containing title and abstract for PubMed ID 1354642

The result is in XML and we can replace the PubMed identifier `1354642` by a comma separated list of identifiers, such as `2298749,1354642,8220422`.

Thus, we can now update the script:

In [ ]:
%%bash
cat > getpublications.sh << 'EOF'
# CHEBI identifier given as input is renamed to ID
ID=$1

# Removes any previous files
rm -f chebi_${ID}_*.rdf

grep -l '<name type="scientific">Homo sapiens</name>' chebi_${ID}_*.xml | \
    xargs -I {} \
      grep '<dbReference type="PubMed"' {} | \
    cut -d'"' -f4 | \
    sort -u | \
    xargs -I {} \
      curl -O 'https://rest.uniprot.org/citations/{}.rdf'
EOF

Again, do not forget to save it in our working directory, and add the right permissions with chmod as we did previously with the other scripts.

In [ ]:
%%bash
chmod u+x getpublications.sh

**Expected Output:** File permissions updated successfully

In [ ]:
%%bash
timeout 10s ./getpublications.sh 27732; echo "Finished or timed out."

**Expected Output:** Downloads multiple RDF files (`chebi_27732_*.rdf`).

> **Note:** We have added a `timeout` because this process takes a significant amount of time due to UniProt API rate limits and access restrictions. Instead of waiting for the script to complete, it is recommended to download the consolidated ZIP file directly from GitHub:

In [ ]:
%%bash
curl -s -O 'https://raw.githubusercontent.com/lasigeBioTM/data-text-processing-notebooks/refs/heads/main/data/chebi_27732_publications_rdf.zip'
unzip -o chebi_27732_publications_rdf.zip

You can verify the files were created by listing them:

In [ ]:
%%bash
ls *.rdf

**Expected Output:** Lists all downloaded RDF files (chebi_27732_*.rdf)

In the next step, we will extract the titles and abstracts from the RDF files.

## Step 02 - Title and Abstract

Each file has the title and abstract of the publication as values of the `title` and `rdfs:comment` elements, respectively. To extract them we can again use the `xmllint` command.
To install `xmllint` we can execute:

In [ ]:
%%bash
apt-get update && apt-get install -y libxml2-utils

 Now we can execute the `xmllint` command:


In [ ]:
%%bash
xmllint --xpath '//*[local-name()="title" or local-name()="comment"]' 10051009.rdf

**Expected Output:** XML elements containing title and comment text

The output should be the text inside XML elements. To remove the XML elements, we can again add `text()` to the XPath query:

In [ ]:
%%bash
xmllint --xpath '//*[local-name()="title" or local-name()="comment"]/text()' 10051009.rdf

**Expected Output:** Clean text of titles and abstracts without XML tags

Let's create the script `gettext.sh`:

In [ ]:
%%bash
cat > gettext.sh << 'EOF'
# CHEBI identifier given as input is renamed to ID
ID=$1

xmllint --xpath '//*[local-name()="title" or local-name()="comment"]/text()' *.rdf
EOF

Again, do not forget to save it in our working directory, and add the right permissions with chmod as we did previously with the other scripts.

In [ ]:
%%bash
chmod u+x gettext.sh

**Expected Output:** File permissions updated successfully

In [ ]:
%%bash
./gettext.sh 27732 | head -n 10

**Expected Output:** Displays the first ten lines of the extracted text

We can save the resulting text in a file named `chebi_27732.txt` that we may share or read using our favorite text editor, by adding the redirection operator:

In [ ]:
%%bash
./gettext.sh 27732 > chebi_27732.txt

**Expected Output:** Creates chebi_27732.txt file with extracted titles and abstracts

In the next step, we will identify diseases in the extracted text.

## Step 03 - Disease Recognition

Instead of reading all that text to find any disease related with caffeine, we can try to find sentences about a given disease by using grep:

In [ ]:
%%bash
grep 'malignant hyperthermia' chebi_27732.txt

**Expected Output:** Lines containing 'malignant hyperthermia' from the text file

To save the filtered text in a file named `chebi_27732_hyperthermia.txt`, we only need to add the redirection operator:

In [ ]:
%%bash
grep 'malignant hyperthermia' chebi_27732.txt > chebi_27732_hyperthermia.txt

**Expected Output:** Creates chebi_27732_hyperthermia.txt with filtered disease mentions

This is a very simple way of recognizing a disease in text. The next tutorials will describe how to perform more complex text processing tasks.

## Conclusion

This concludes the **Unix Shell** tutorial adapted from the same section of the [Data and Text Processing for Health and Life Sciences](https://labs.rd.ciencias.ulisboa.pt/book/) book.

In this tutorial, we learned how to retrieve protein-related publications, extract text from titles and abstracts, and identify mentioned diseases.

In the next tutorial in this series will explore more efficient pattern matching techniques to identify diseases in the text.

## Exercise 01

As an exercise, try to identify another disease in the file `chebi_27732.txt`.